# Data Cleaning 01 -- Top 100 S&P 500 Universe (Master List)

## Input
`Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet` (single column: `permno`)

## Purpose
The master list contains all unique PERMNOs that ever appeared in any year's top-100 S&P 500 universe. It should be exactly the set of distinct PERMNOs from `universe_annual.parquet` -- nothing more, nothing less. This notebook validates that property and either copies the file unchanged or regenerates it from the annual file.

## Stage 0: Load & Inspect
Shape, dtype, and PERMNO range verification.

## Stage 1: Missing Data Audit
- Total NaN count
- Per-column NaN counts
- Duplicate PERMNO check (the file should contain only unique values)

## Stage 2: Cross-Check Against Annual Universe
The master PERMNO set is compared against the set of unique PERMNOs extracted from the already-cleaned `universe_annual_clean.parquet`:
- Identifies orphan PERMNOs (in master but not in annual)
- Identifies missing PERMNOs (in annual but not in master)
- Confirms or flags whether the two sets match exactly
- Reports the distribution of how many years each PERMNO appears in the annual file (all 21 years, 10--20, 5--9, 1--4)

## Stage 3: Save
If all checks pass (no NaN, no duplicates, exact set match with annual), the raw file is copied directly. If any mismatch is detected, the master list is regenerated from the annual file to ensure consistency.

## Output
`Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet`

In [1]:
# %% [markdown]
# # Data Cleaning: universe_master.parquet
#
# Source: Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet
# Output: Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet
#
# The master list contains all unique PERMNOs that ever appeared in any year's
# top-100 S&P 500 universe. It should be exactly the set of distinct PERMNOs
# from universe_annual.parquet — nothing more, nothing less.
 
# %%
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
 
RAW_PATH     = Path('../../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet')
ANNUAL_PATH  = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
OUT_DIR      = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe')
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════
 
# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — universe_master")
print("=" * 90)
 
master = pd.read_parquet(RAW_PATH)
 
print(f"\nShape: {master.shape[0]:,} rows × {master.shape[1]} columns")
 
print(f"\nColumns and dtypes:")
for c in master.columns:
    print(f"  {c:<15s} {str(master[c].dtype)}")
 
print(f"\nPERMNO range: {master['permno'].min()} → {master['permno'].max()}")
 
print(f"\n--- Head (10 rows) ---")
print(master.head(10).to_string(index=False))
 
print(f"\n--- Tail (10 rows) ---")
print(master.tail(10).to_string(index=False))
 

STAGE 0: LOAD & INSPECT — universe_master

Shape: 227 rows × 1 columns

Columns and dtypes:
  permno          Int64

PERMNO range: 10104 → 93436

--- Head (10 rows) ---
 permno
  12060
  10107
  11850
  21936
  70519
  55976
  59328
  66800
  76076
  12490

--- Tail (10 rows) ---
 permno
  60871
  36468
  86339
  24766
  18576
  76744
  11762
  64390
  92108
  13511


In [2]:
 
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════
 
# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)
 
# ── Total NaN ────────────────────────────────────────────────────────────────
total_nan = master.isna().sum().sum()
print(f"\nTotal NaN: {total_nan}")
 
# ── Per-column NaN ───────────────────────────────────────────────────────────
for c in master.columns:
    n = master[c].isna().sum()
    status = "✓" if n == 0 else "⚠"
    print(f"  {status} {c:<15s} {n} NaN")
 
# ── Duplicates ───────────────────────────────────────────────────────────────
n_dupes = master['permno'].duplicated().sum()
print(f"\nDuplicate PERMNOs: {n_dupes}")
if n_dupes > 0:
    dupes = master[master['permno'].duplicated(keep=False)]
    print(f"  ⚠ Duplicate values:")
    print(dupes.to_string(index=False))


STAGE 1: MISSING DATA AUDIT

Total NaN: 0
  ✓ permno          0 NaN

Duplicate PERMNOs: 0


In [3]:
 
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: CROSS-CHECK AGAINST ANNUAL UNIVERSE
# ═══════════════════════════════════════════════════════════════════════════════
 
# %%
print("\n" + "=" * 90)
print("STAGE 2: CROSS-CHECK AGAINST universe_annual_clean")
print("=" * 90)
 
annual = pd.read_parquet(ANNUAL_PATH)
 
master_set = set(master['permno'])
annual_set = set(annual['permno'].unique())
 
print(f"\n  PERMNOs in master:          {len(master_set)}")
print(f"  Unique PERMNOs in annual:   {len(annual_set)}")
 
in_master_not_annual = sorted(master_set - annual_set)
in_annual_not_master = sorted(annual_set - master_set)
 
print(f"\n  In master but NOT in annual: {len(in_master_not_annual)}")
if in_master_not_annual:
    print(f"    ⚠ Orphan PERMNOs: {in_master_not_annual}")
 
print(f"  In annual but NOT in master: {len(in_annual_not_master)}")
if in_annual_not_master:
    print(f"    ⚠ Missing PERMNOs: {in_annual_not_master}")
 
if master_set == annual_set:
    print(f"\n  ✓ Perfect match — master contains exactly the unique PERMNOs from annual")
else:
    print(f"\n  ⚠ MISMATCH — master and annual have different PERMNO sets")
    print(f"    Recommendation: regenerate master from annual rather than copying raw file")
 
# ── How many years does each PERMNO appear? ──────────────────────────────────
appearances = annual.groupby('permno')['year'].nunique().sort_values(ascending=False)
print(f"\n  Appearance distribution:")
print(f"    In all 21 years:  {(appearances == 21).sum()}")
print(f"    10-20 years:      {((appearances >= 10) & (appearances < 21)).sum()}")
print(f"    5-9 years:        {((appearances >= 5) & (appearances < 10)).sum()}")
print(f"    1-4 years:        {(appearances < 5).sum()}")
print(f"    Median years:     {appearances.median():.0f}")
 



STAGE 2: CROSS-CHECK AGAINST universe_annual_clean

  PERMNOs in master:          227
  Unique PERMNOs in annual:   227

  In master but NOT in annual: 0
  In annual but NOT in master: 0

  ✓ Perfect match — master contains exactly the unique PERMNOs from annual

  Appearance distribution:
    In all 21 years:  40
    10-20 years:      52
    5-9 years:        50
    1-4 years:        85
    Median years:     7


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SAVE
# ═══════════════════════════════════════════════════════════════════════════════
 
# %%
print("\n" + "=" * 90)
print("STAGE 3: SAVE")
print("=" * 90)
 
out_path = OUT_DIR / 'universe_master_clean.parquet'
 
if master_set == annual_set and total_nan == 0 and n_dupes == 0:
    # File is clean — straight copy
    shutil.copy2(RAW_PATH, out_path)
    print(f"\n  ✓ File is clean — copied directly to {out_path}")
    print(f"    {len(master)} unique PERMNOs")
else:
    # Regenerate from annual to ensure consistency
    master_regen = pd.DataFrame({'permno': sorted(annual['permno'].unique())})
    master_regen.to_parquet(out_path, index=False, engine='pyarrow')
    print(f"\n  ⚠ Regenerated master from annual to fix mismatches")
    print(f"    Saved: {out_path}")
    print(f"    {len(master_regen)} unique PERMNOs")
 
print("\nCleaning complete.")


STAGE 3: SAVE

  ✓ File is clean — copied directly to ..\..\..\Data\Data_Collection\Cleaned\01_Top100_SP500_Universe\universe_master_clean.parquet
    227 unique PERMNOs

Cleaning complete.
